# Cleaning

The cleaning is done by `load_text()` in `src/data.py`. This notebook runs it and checks the result.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
from src.data import clean_text, load_text

In [2]:
raw = pd.read_parquet("../data/complaints.parquet")
raw.shape

(432952, 18)

## Rows kept

In [3]:
has_text = raw["Consumer complaint narrative"].notna()
labelled = raw["Consumer disputed?"] != "N/A"
in_window = raw["Date received"] <= "2017-02-28"
has_text.sum(), (has_text & labelled).sum(), (has_text & labelled & in_window).sum()

(np.int64(164437), np.int64(164003), np.int64(149661))

In [4]:
recent = raw[has_text & labelled & (raw["Date received"] >= "2017-01-01")]
(recent["Consumer disputed?"] == "Yes").groupby(recent["Date received"].dt.to_period("W")).agg(["size", "mean"]).round(3)

,size,mean
Date received,,
2016-12-26/2017-01-01,60,0.183
2017-01-02/2017-01-08,1578,0.229
2017-01-09/2017-01-15,1656,0.205
2017-01-16/2017-01-22,3132,0.185
2017-01-23/2017-01-29,2394,0.207
2017-01-30/2017-02-05,2019,0.218
2017-02-06/2017-02-12,2042,0.192
2017-02-13/2017-02-19,1876,0.192
2017-02-20/2017-02-26,1749,0.192


Same pattern as in the complaint-escalation project. The weekly rate holds at about 19 to 23% until 12 Mar 2017 and then falls to 1.6% in the last week, because disputes stopped being recorded on 24 Apr 2017. The same cut-off, 28 Feb 2017, leaves 149,661 complaints.

## Redaction marks

In [5]:
narrative = raw.loc[has_text, "Consumer complaint narrative"]
narrative.str.findall(r"[X/]*X{2,}[X/\d]*").explode().value_counts().head(12)

Consumer complaint narrative
XXXX              1271356
XX/XX/XXXX          82428
XXXX/XXXX/XXXX      20099
XX/XX/2016           8616
XXXX/XXXX/16         6374
XXXX/XXXX/2016       5642
XXXX/XXXX/15         5507
XXXX/XXXX/2015       4918
XX/XX/2017           3884
XX/XX/2015           3100
XXXXXXXXXXXX         2332
XXXX/                2004
Name: count, dtype: int64

In [6]:
narrative.str.findall(r"\{\$[^}]*\}").explode().value_counts().head(5)

Consumer complaint narrative
{$1000.00}    4561
{$100.00}     3601
{$500.00}     3513
{$200.00}     2651
{$0.00}       2617
Name: count, dtype: int64

There are three kinds: dates, amounts in braces, and XXXX for everything else. `clean_text()` turns each kind into one token.

In [7]:
example = narrative.head(1)
print(example.iloc[0])
print()
print(clean_text(example).iloc[0])

I am a customer of Rushcard. On XXXX/XXXX/15 my employer direct deposited {$400.00}, however, on XXXX/XXXX/15 there was a deposit reversal done by this company not my employer. I simply want my money back. It is mid XXXX and I still cant get through to customer service and I have no idea how to get information on my funds.

I am a customer of Rushcard. On xxdate my employer direct deposited xxmoney , however, on xxdate there was a deposit reversal done by this company not my employer. I simply want my money back. It is mid xxxx and I still cant get through to customer service and I have no idea how to get information on my funds.


## Clean table

In [8]:
df = load_text()
df.shape

(149661, 15)

In [9]:
df.head(3)

,complaint_id,date_received,date_sent,product,sub_product,issue,sub_issue,company,tags,company_response,timely_response,narrative,disputed,text,copies
0,1665710,2015-11-20,2015-11-24,Prepaid card,Transit card,Unauthorized transactions/trans. issues,None,"Empowerment Ventures, LLC",None,Closed with monetary relief,Yes,I am a customer of Rushcard. On XXXX/XXXX/15 m...,1,I am a customer of Rushcard. On xxdate my empl...,1
1,1934081,2016-05-20,2016-05-20,Debt collection,"Other (i.e. phone, health club, etc.)",Improper contact or sharing of info,Talked to a third party about my debt,"Williams & Fudge, Inc",None,Closed with explanation,Yes,"This individual contacted my father, who is in...",1,"This individual contacted my father, who is in...",1
2,1424944,2015-06-17,2015-06-17,Credit reporting,None,Incorrect information on credit report,Information is not mine,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",None,Closed with explanation,Yes,i entered disputes with all XXXX bureaus. i do...,0,i entered disputes with all xxxx bureaus. i do...,1


In [10]:
df.dtypes

complaint_id                 int64
date_received       datetime64[us]
date_sent           datetime64[us]
product                     string
sub_product                 string
issue                       string
sub_issue                   string
company                     string
tags                        string
company_response            string
timely_response             string
narrative                      str
disputed                     int64
text                           str
copies                       int64
dtype: object

In [11]:
df.isna().sum().sum()

np.int64(0)

In [12]:
df["date_received"].min(), df["date_received"].max()

(Timestamp('2015-03-19 00:00:00'), Timestamp('2017-02-28 00:00:00'))

In [13]:
df["disputed"].mean()

np.float64(0.22640500865288887)

In [14]:
df["text"].str.contains("X{2,}").sum()

np.int64(0)

In [15]:
for token in ["xxxx", "xxdate", "xxmoney"]:
    print(token, round(df["text"].str.contains(token).mean(), 3))

xxxx 0.812
xxdate 0.278
xxmoney 0.308


In [16]:
df["product"].value_counts()

product
Debt collection            36751
Mortgage                   30707
Credit reporting           28329
Credit card                17333
Bank account or service    13734
Student loan                9585
Consumer Loan               8565
Payday loan                 1632
Money transfers             1382
Prepaid card                1369
Other financial service      259
Virtual currency              15
Name: count, dtype: int64[pyarrow]

In [17]:
df["company"].nunique()

3032

## Short narratives

In [18]:
words = df["text"].str.replace("xxxx|xxdate|xxmoney", " ", regex=True).str.split().str.len()
words.min(), (words < 5).sum(), (words < 10).sum()

(np.int64(1), np.int64(149), np.int64(1193))

In [19]:
df.loc[words < 3, "text"].head(10)

17319               daily calls
29098       Threats, harassment
33346    incorrect informations
45104     Fraudulent inquiries.
72827            Wrong address.
73015                  Not sure
82497            Repeated calls
83019                 fradulent
84552             VIOLATED FCRA
88224             VIOLATED FCRA
Name: text, dtype: str

In [20]:
df.groupby(pd.cut(words, [0, 4, 9, 24, 2000]))["disputed"].agg(["size", "mean"]).round(3)

,size,mean
text,,
"(0, 4]",149,0.094
"(4, 9]",1044,0.131
"(9, 24]",6978,0.159
"(24, 2000]",141490,0.231


None is empty after cleaning, so they all stay. The shortest ones are disputed far less often, 9.4% under 5 words against 23.1% for 25 or more.

## Repeated texts

In [21]:
(df["copies"] > 1).sum(), df["text"].str.lower().duplicated().sum()

(np.int64(4753), np.int64(3010))

In [22]:
df.groupby(pd.cut(df["copies"], [0, 1, 2, 9, 200]))["disputed"].agg(["size", "mean"]).round(3)

,size,mean
copies,,
"(0, 1]",144908,0.228
"(1, 2]",2544,0.180
"(2, 9]",1563,0.193
"(9, 200]",646,0.076


In [23]:
repeats = df[df["copies"] > 1]
same_text = repeats.groupby(repeats["text"].str.lower())
(same_text["company"].nunique() > 1).mean(), (same_text["disputed"].nunique() > 1).mean()

(np.float64(0.7831325301204819), np.float64(0.19334480780263913))

In [24]:
train = df[df["date_received"] <= "2016-09-30"]
test = df[df["date_received"] > "2016-09-30"]
len(test), test["text"].str.lower().isin(train["text"].str.lower()).sum()

(37405, np.int64(77))

Texts with 10 or more copies are almost never disputed (7.6%). 78% of the repeated texts went to more than one company, probably one consumer complaining about several companies at once, though there's no consumer ID to check. Only 77 of the 37,405 complaints after Sep 2016 repeat a text from before, so the copies stay in and `copies` keeps track of them.